# Clean PCA Source Summary CSV

This notebook loads `pca_source_rank_weight_summary.csv`, applies requested cleaning rules, and writes a cleaned summary CSV.

In [5]:
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'src').exists() and (p / 'outputs').exists():
            return p
    return start

ROOT = find_project_root(Path.cwd().resolve())
input_path = ROOT / 'outputs' / 'pca_source_rank_weight_test2' / 'pca_source_rank_weight_summary.csv'
output_path = input_path.with_name('pca_source_rank_weight_summary_cleaned.csv')

if not input_path.exists():
    raise FileNotFoundError(f'Input file not found: {input_path}')

print('ROOT:', ROOT)
print('Input:', input_path)
print('Output:', output_path)

ROOT: C:\Users\Simen\OneDrive - NTNU\FYSMAT\INDMAT\MASTER\Master_thesis
Input: C:\Users\Simen\OneDrive - NTNU\FYSMAT\INDMAT\MASTER\Master_thesis\outputs\pca_source_rank_weight_test2\pca_source_rank_weight_summary.csv
Output: C:\Users\Simen\OneDrive - NTNU\FYSMAT\INDMAT\MASTER\Master_thesis\outputs\pca_source_rank_weight_test2\pca_source_rank_weight_summary_cleaned.csv


In [6]:
df = pd.read_csv(input_path)
required_cols = {
    'trait', 'target_island', 'target_island_name', 'analysis', 'method',
    'n_components', 'n_individuals', 'corr_mean', 'corr_std', 'mse_mean', 'n_rows'
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Missing expected columns: {sorted(missing)}')

print('Rows before cleaning:', len(df))
df.head()

Rows before cleaning: 291


,trait,target_island,target_island_name,analysis,method,n_components,alpha,n_individuals,corr_mean,corr_std,mse_mean,n_rows
0,body_mass,0,Nesøy,distance_threshold,pca_source_alpha,10.0,1.0,186,0.007843,NaN,1.304924,1
1,body_mass,0,Nesøy,distance_threshold,pca_source_alpha,10.0,1.5,446,0.277811,NaN,1.239449,1
2,body_mass,0,Nesøy,distance_threshold,pca_source_alpha,10.0,2.0,794,0.240097,NaN,1.252944,1
3,body_mass,0,Nesøy,distance_threshold,pca_source_alpha,10.0,3.0,1462,0.231369,NaN,1.265889,1
4,body_mass,0,Nesøy,distance_threshold,pca_source_alpha,10.0,4.0,2085,0.187833,NaN,1.282366,1


In [7]:
clean = df.copy()
clean['n_components'] = pd.to_numeric(clean['n_components'], errors='coerce')

# Rule 1: full_source_unweighted should only have one row per island/trait (PC not used).
full_mask = clean['method'].eq('full_source_unweighted')
full_rows = clean[full_mask].copy()
non_full_rows = clean[~full_mask].copy()

full_dedup_subset = [
    c for c in full_rows.columns
    if c != 'n_components'
]
full_rows = full_rows.drop_duplicates(subset=full_dedup_subset).copy()
full_rows['n_components'] = pd.NA

clean = pd.concat([non_full_rows, full_rows], ignore_index=True)

# Rule 2: drop the largest n_individuals row per island+PC for ranked_subset
# because it matches full_source_unweighted (same training set).
ranked_mask = clean['analysis'].eq('ranked_subset') & clean['n_individuals'].notna()
ranked = clean[ranked_mask].copy()
other = clean[~ranked_mask].copy()

group_cols = [
    c for c in ['trait', 'target_island', 'target_island_name', 'n_components']
    if c in ranked.columns
]
ranked['max_n_in_group'] = ranked.groupby(group_cols)['n_individuals'].transform('max')
ranked = ranked[ranked['n_individuals'] != ranked['max_n_in_group']].drop(columns=['max_n_in_group'])

clean = pd.concat([other, ranked], ignore_index=True)

# Rule 3: random rows are duplicated across PCs. Keep one row and empty n_components.
rand_mask = clean['method'].eq('random_individual')
rand_rows = clean[rand_mask].copy()
non_rand_rows = clean[~rand_mask].copy()

rand_dedup_subset = [
    c for c in rand_rows.columns
    if c != 'n_components'
]
rand_rows = rand_rows.drop_duplicates(subset=rand_dedup_subset).copy()
rand_rows['n_components'] = pd.NA

clean = pd.concat([non_rand_rows, rand_rows], ignore_index=True)

# Final ordering for readability and stable output.
sort_cols = [
    c for c in ['trait', 'target_island', 'target_island_name', 'analysis', 'method', 'n_components', 'n_individuals']
    if c in clean.columns
]
clean = clean.sort_values(sort_cols, na_position='first').reset_index(drop=True)

# Keep n_components empty for rows where it is not relevant.
clean['n_components'] = clean['n_components'].astype('Int64').astype('string')
clean.loc[clean['n_components'] == '<NA>', 'n_components'] = ''

print('Rows after cleaning:', len(clean))
clean.head(20)

Rows after cleaning: 280


,trait,target_island,target_island_name,analysis,method,n_components,alpha,n_individuals,corr_mean,corr_std,mse_mean,n_rows
0,body_mass,0,Nesøy,distance_threshold,pca_source_alpha,10,1.0,186,0.007843,NaN,1.304924,1
1,body_mass,0,Nesøy,distance_threshold,pca_source_alpha,10,1.5,446,0.277811,NaN,1.239449,1
2,body_mass,0,Nesøy,distance_threshold,pca_source_alpha,10,2.0,794,0.240097,NaN,1.252944,1
3,body_mass,0,Nesøy,distance_threshold,pca_source_alpha,10,3.0,1462,0.231369,NaN,1.265889,1
4,body_mass,0,Nesøy,distance_threshold,pca_source_alpha,10,4.0,2085,0.187833,NaN,1.282366,1
5,body_mass,0,Nesøy,full_baseline,full_source_unweighted,<NA>,NaN,3326,0.210720,NaN,1.285977,1
6,body_mass,0,Nesøy,ranked_subset,pca_source_topk,10,NaN,50,0.136888,NaN,1.293928,1
7,body_mass,0,Nesøy,ranked_subset,pca_source_topk,10,NaN,200,0.131773,NaN,1.298411,1
8,body_mass,0,Nesøy,ranked_subset,pca_source_topk,10,NaN,500,0.265117,NaN,1.241223,1
9,body_mass,0,Nesøy,ranked_subset,pca_source_topk,10,NaN,1000,0.241479,NaN,1.254307,1


In [8]:
report = pd.DataFrame([
    {'metric': 'rows_before', 'value': len(df)},
    {'metric': 'rows_after', 'value': len(clean)},
    {'metric': 'rows_removed', 'value': len(df) - len(clean)},
])

clean.to_csv(output_path, index=False)
print('Saved cleaned summary:', output_path)

report

Saved cleaned summary: C:\Users\Simen\OneDrive - NTNU\FYSMAT\INDMAT\MASTER\Master_thesis\outputs\pca_source_rank_weight_test2\pca_source_rank_weight_summary_cleaned.csv


,metric,value
0,rows_before,291
1,rows_after,280
2,rows_removed,11
